# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d09_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # water_quality_ewqi train
    shutil.copy(glob.glob("/kaggle/input/**/d09_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # water_quality_ewqi test
    print("Kaggle: water_quality_ewqi data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: water_quality_ewqi data copied to ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795364165.118807      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795364185.035262      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (water_quality_ewqi_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['pH', 'TDS', 'EC', 'TH', 'ca', 'mg', 'CO3', 'HCO3', 'NA', 'K', 'CHLORIDE', 'SULPHATE', 'NITRATE', 'FLUORIDE']
TARGET_LABEL = 'Pre-monsoon EWQI'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (47, 15)
First 5 rows of training data:
      pH   TDS    EC    TH   ca    mg  CO3  HCO3   NA    K  CHLORIDE  SULPHATE  \
0  7.90   552   865   452  272   180  0.2   4.5  260   44       375       681   
1  7.52  1985  2256  1054  529   525  0.0  15.0  272   47       728       324   
2  7.65  1865  2032   636  371   265  0.0   9.0  373   25       713       503   
3  7.25  4855  7856  2050  795  1255  0.0  12.5  335  106        81       676   
4  7.96  2855  3696   865  310   555  0.3   4.6  184  106       698       324   

   NITRATE  FLUORIDE  Pre monsoon EWQI   
0       60      0.40              70.52  
1       10      0.65             138.78  
2       54      0.85             113.07  
3       17      0.69             266.32  
4       25      1.20             158.05  

Shape of test data: (47, 15)
First 5 rows of test data:
      pH   TDS    EC    TH     ca      mg  CO3  HCO3   NA    K  CHLORIDE  \
0  8.38   625   975   320  120.0   200.0  0.0   5.8  280   35 

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (47, 14)
Shape of y_train: (47,)
Shape of X_test: (47, 14)
Shape of y_test: (47,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 0.3797749  -1.15230942 -0.94309903 -0.47923131 -0.09229732 -0.56852482
  -0.03922784 -1.18768739 -0.02713192 -0.87884263 -0.2770724   1.15873613
   1.41276153 -1.25316406]
 [-0.78765707 -0.2953837  -0.52059702  0.45803078  1.31357822  0.10564733
  -0.50015494  1.61405127  0.07159325 -0.77136925  1.29937074 -0.45273785
  -1.61366471 -0.53733141]
 [-0.38827245 -0.36714301 -0.58863473 -0.19275917  0.44926563 -0.40242444
  -0.50015494  0.01305775  0.90253007 -1.55950739  1.23238307  0.35525611
   1.04959038  0.03533472]
 [-1.61714822  1.42085972  1.18034563  2.00871688  2.76868675  1.53215652
  -0.50015494  0.94697063  0.58990037  1.34227396 -1.59003071  1.13616647
  -1.18996504 -0.42279818]
 [ 0.56410627  0.22487128 -0.08321177  0.16377408  0.11557533  0.16427099
   0.19123571 -1.16100417 -0.6523913   1.34227396  1.1653954  -0.45273785
  -0.70573684  1.03750043]]

First five rows of normalized X_test:
[[ 1.85442582 -1.10865585 -0.90968766 -0.6847

# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/water-quality-ewqi-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 815.3942506476026,
  'cv_rmse': 28.555109011306584,
  'best_params': {'n_estimators': 200,
   'criterion': 'squared_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': 200,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.1},
  'test_mse': 517.6088147057782,
  'test_rmse': np.float64(22.751017882850388),
  'test_corr_coef': np.float64(0.9547680128224817),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 800.6499765894229,
  'cv_rmse': 28.295758985922657,
  'best_params': {'n_estimators': 300,
   'criterion': 'squared_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 0.01,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0   67.517536   67.508847   70.246859   71.119727   77.957014   75.436182   
1  110.210677  122.738840  125.843652  132.977329  137.370214  141.773023   
2   87.734377   87.609843   88.882805   89.441509   92.541371   87.951844   
3  113.491346  127.867623  199.066493  202.852373  225.821945  236.344640   
4  108.474685  130.235272  135.871331  144.972469  154.100557  155.044135   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   79.880743   83.217884   94.753168  113.832465  119.408621  123.661119   
1  141.561582  135.396888  142.857562  134.240168  137.356581  138.297436   
2   89.872687   95.546327   92.825840  100.248882  104.030448  110.230830   
3  235.474961  256.569969  263.885794  265.947430  265.536268  266.578205   
4  158.265146  161.444217  162.002476  166.371572  166.633450  161.319331   

         0.95  Actual  
0  117.877589   73.07  
1  144.297508  141.67  
2 

In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 53.19%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 46 of 47 rows (97.87%)


{'empirical_quantiles': [np.float64(0.0425531914893617),
  np.float64(0.06382978723404255),
  np.float64(0.10638297872340426),
  np.float64(0.10638297872340426),
  np.float64(0.1702127659574468),
  np.float64(0.1702127659574468),
  np.float64(0.1702127659574468),
  np.float64(0.2553191489361702),
  np.float64(0.3191489361702128),
  np.float64(0.3829787234042553),
  np.float64(0.46808510638297873),
  np.float64(0.5319148936170213),
  np.float64(0.574468085106383)],
 'average_interval_lengths': [np.float64(18.7404888855848),
  np.float64(43.845257450985315),
  np.float64(56.42582706453798)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.076497115725759),
 'average_crps': np.float64(14.36727829863594),
 'nll_values': array([ 3.83581181,  3.34369826,  5.43643865,  5.1172671 ,  7.18606207,
         6.03575629,  4.15124515,  4.17283148,  4.56321082,  3.75364481,
         6.80851248,  3.74602744,  3.40050217,  3.20043029,  3.94275417,
        13.23899891,  4.3726674 , 12.4

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 45.83%
The empirical coverage after calibration is: 100.00%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

        0.05         0.1        0.15         0.2         0.3         0.4  \
0  69.645630   66.434540   63.470806   67.762848   73.504059   80.163277   
1  72.104065  100.366089  120.175972  129.913284  139.491486  138.213928   
2  72.104065   91.159149   95.349892   87.397957   85.851692   85.244766   
3  72.104065  101.027130  121.883186  158.549393  213.094894  213.826355   
4  72.104065  100.146072  116.953384  140.961563  155.569809  152.401810   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   76.173561   77.655510   91.967896  105.821159  114.288887  118.512856   
1  148.243546  147.272736  139.024765  138.692108  142.758331  167.492462   
2   83.623550   97.685196   93.367554  105.866570  112.920303  117.424744   
3  243.511093  253.277466  266.655640  282.706421  272.442017  277.053467   
4  164.176437  170.626556  165.830261  184.476868  174.471344  198.494720   

         0.95  Actual  
0  203.278595   73.07  
1  203.278595  141.67  
2  203.2

In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 93.62%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 47 of 47 rows (100.0%)


{'empirical_quantiles': [np.float64(0.02127659574468085),
  np.float64(0.02127659574468085),
  np.float64(0.0425531914893617),
  np.float64(0.06382978723404255),
  np.float64(0.10638297872340426),
  np.float64(0.14893617021276595),
  np.float64(0.1702127659574468),
  np.float64(0.23404255319148937),
  np.float64(0.3404255319148936),
  np.float64(0.48936170212765956),
  np.float64(0.5531914893617021),
  np.float64(0.6808510638297872),
  np.float64(0.9574468085106383)],
 'average_interval_lengths': [np.float64(34.96821448143492),
  np.float64(69.12590132368372),
  np.float64(142.7412317965893)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.808243268990589),
 'average_crps': np.float64(14.349876942289633),
 'nll_values': array([4.67422949, 4.60686914, 4.69632807, 5.19915734, 5.06822478,
        5.91552501, 4.71313227, 4.62838226, 4.9187534 , 4.60595659,
        5.14746567, 4.63516671, 4.65063489, 4.60642695, 4.67751911,
        4.69166812, 4.73538096, 4.77151831, 4.68

In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 91.67%
The empirical coverage after calibration is: 95.83%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0   68.644049   68.270485   73.155541   72.095268   77.510595   80.644687   
1  103.365573  121.526277  131.873393  134.192506  137.343073  139.206617   
2   88.118407   87.033195   89.812261   87.618447   92.267696   86.626244   
3  110.123973  180.381080  186.058162  212.937911  227.689703  241.010523   
4  103.336495  134.271802  149.411715  151.923081  159.582853  165.514273   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   87.273030   96.111397  104.879463  104.597874  114.447651  116.499475   
1  135.274197  140.084068  132.531030  137.420301  137.466178  139.696870   
2   86.857713   94.785670   97.158610  102.018062  105.930264  110.478616   
3  253.907676  265.607935  266.433889  266.064942  276.462548  281.155557   
4  164.750817  162.097912  159.371499  176.200584  165.346179  168.468302   

         0.95  Actual  
0  125.310363   73.07  
1  162.399415  141.67  
2 

In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 65.96%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 47 of 47 rows (100.0%)


{'empirical_quantiles': [np.float64(0.02127659574468085),
  np.float64(0.0425531914893617),
  np.float64(0.0851063829787234),
  np.float64(0.14893617021276595),
  np.float64(0.1702127659574468),
  np.float64(0.1702127659574468),
  np.float64(0.2127659574468085),
  np.float64(0.2765957446808511),
  np.float64(0.3404255319148936),
  np.float64(0.3617021276595745),
  np.float64(0.5531914893617021),
  np.float64(0.574468085106383),
  np.float64(0.6808510638297872)],
 'average_interval_lengths': [np.float64(19.03530208038364),
  np.float64(37.79898678644432),
  np.float64(66.42492821113642)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.522668546918534),
 'average_crps': np.float64(12.94775517580574),
 'nll_values': array([4.10738813, 3.83532059, 4.66429304, 4.95236263, 5.68616756,
        5.6882904 , 4.28670674, 3.81292072, 4.66446325, 3.7771709 ,
        6.0996086 , 3.92889623, 3.56229642, 3.57904219, 4.03787442,
        5.61622703, 4.03977246, 7.42993555, 4.23342269,

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 58.33%
The empirical coverage after calibration is: 100.00%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=5.5294 val_loss=0.0000 scale=1.0000 norm=46.2626


[iter 100] loss=-0.6880 val_loss=0.0000 scale=1.0000 norm=0.3488


[iter 200] loss=-3.9635 val_loss=0.0000 scale=1.0000 norm=0.3643


[iter 300] loss=-6.8062 val_loss=0.0000 scale=1.0000 norm=0.2975


[iter 400] loss=-9.5524 val_loss=0.0000 scale=1.0000 norm=0.2770


         0.05         0.1        0.15         0.2         0.3         0.4  \
0   69.094410   69.094414   69.094417   69.094419   69.094423   69.094426   
1  136.409190  136.409191  136.409192  136.409192  136.409193  136.409194   
2   86.083974   86.083975   86.083975   86.083975   86.083976   86.083977   
3  261.169206  261.169209  261.169211  261.169212  261.169215  261.169217   
4  156.760320  156.760328  156.760333  156.760337  156.760343  156.760349   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   69.094429   69.094432   69.094435   69.094439   69.094441   69.094444   
1  136.409195  136.409195  136.409196  136.409197  136.409198  136.409198   
2   86.083977   86.083978   86.083978   86.083979   86.083979   86.083980   
3  261.169219  261.169221  261.169223  261.169226  261.169227  261.169229   
4  156.760354  156.760359  156.760365  156.760371  156.760375  156.760380   

         0.95  Actual  
0   69.094449   73.07  
1  136.409199  141.67  
2 

In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 0.00%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 47 rows (0.0%)


{'empirical_quantiles': [np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937),
  np.float64(0.23404255319148937)],
 'average_interval_lengths': [np.float64(1.476046578194984e-05),
  np.float64(2.247602279528516e-05),
  np.float64(2.8847662942185475e-05)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(23626678997949.88),
 'average_crps': np.float64(14.766435711925197),
 'nll_values': array([5.76195301e+10, 1.88394854e+12, 7.52916854e+13, 1.48182498e+12,
        2.49251755e+12, 1.40303828e+11, 1.38988404e+12, 1.36905196e+11,
        8.52746717e+13, 1.47162851e+13, 1.78839191e+11, 1.39209103e+13,
        2.4070340

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 0.00%
The empirical coverage after calibration is: 100.00%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

    0.05    0.1   0.15   0.2    0.3        0.4     0.5      0.6       0.7  \
0  72.97  74.95  80.98  82.5  93.56   93.15000  106.16  113.070  129.8300   
1  72.97  74.95  80.98  82.5  93.56  112.47216  119.14  128.298  148.7360   
2  72.97  74.95  80.98  82.5  93.56   93.15000  106.16  113.070  129.8300   
3  72.97  74.95  80.98  82.5  93.56  112.47216  119.14  128.298  173.1488   
4  72.97  74.95  80.98  82.5  93.56  112.47216  119.14  128.298  148.7360   

         0.8     0.85     0.9    0.95  Actual  
0  139.89384  169.134  205.93  255.83   73.07  
1  139.89384  169.134  205.93  255.83  141.67  
2  139.89384  169.134  205.93  255.83  110.44  
3  205.93000  205.930  205.93  255.83  275.01  
4  139.89384  169.134  205.93  255.83  202.57  


In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 89.36%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 38 of 47 rows (80.85%)


{'empirical_quantiles': [np.float64(0.02127659574468085),
  np.float64(0.0425531914893617),
  np.float64(0.0425531914893617),
  np.float64(0.0425531914893617),
  np.float64(0.14893617021276595),
  np.float64(0.2127659574468085),
  np.float64(0.3191489361702128),
  np.float64(0.3829787234042553),
  np.float64(0.46808510638297873),
  np.float64(0.5957446808510638),
  np.float64(0.7872340425531915),
  np.float64(0.8723404255319149),
  np.float64(0.9148936170212766)],
 'average_interval_lengths': [np.float64(68.73406130269076),
  np.float64(130.98000000563317),
  np.float64(182.85999999757698)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.572662700963716),
 'average_crps': np.float64(30.048034534727197),
 'nll_values': array([ 5.11405136,  5.01900358,  4.93982517,  8.86848634,  6.06325799,
        12.18869328,  4.94467736,  4.93698047,  6.92532818,  4.94479024,
         5.52503574,  4.96907897,  5.01834862,  4.96028032,  5.29677141,
         5.00667118,  5.24572545,  

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 83.33%
The empirical coverage after calibration is: 83.33%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0   85.372796   80.974515   81.867505   82.004920   85.762538   78.061975   
1  130.294399  126.153216  123.604717  125.045701  134.687154  129.467759   
2   96.253864  103.096337   96.602422   98.449770   90.369950   89.182990   
3  246.222737  229.905499  208.583828  226.692236  256.025997  254.278299   
4  172.569302  168.410625  170.877889  186.825730  160.681998  174.927606   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   86.004158   88.227986   83.036472   93.663994   91.830833   87.090861   
1  142.447270  138.754441  142.511055  133.917776  135.460005  131.046830   
2   85.642465  100.892959  102.634970   99.728625   94.410782  112.582891   
3  247.333888  259.732467  267.109344  260.522878  249.969958  257.278189   
4  167.627325  147.301720  152.515204  149.189897  151.440791  149.786092   

         0.95  Actual  
0   77.105122   73.07  
1  165.121660  141.67  
2 

In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 63.83%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 47 of 47 rows (100.0%)


{'empirical_quantiles': [np.float64(0.06382978723404255),
  np.float64(0.06382978723404255),
  np.float64(0.0851063829787234),
  np.float64(0.1276595744680851),
  np.float64(0.14893617021276595),
  np.float64(0.23404255319148937),
  np.float64(0.2765957446808511),
  np.float64(0.3191489361702128),
  np.float64(0.3617021276595745),
  np.float64(0.425531914893617),
  np.float64(0.48936170212765956),
  np.float64(0.5957446808510638),
  np.float64(0.7021276595744681)],
 'average_interval_lengths': [np.float64(13.10203344284313),
  np.float64(25.148747140133086),
  np.float64(36.411691255288154)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.876415179502655),
 'average_crps': np.float64(12.639221739421897),
 'nll_values': array([ 5.52203056,  3.6429044 ,  3.96213265,  4.54432295,  8.54194112,
         7.43292101,  4.73188824,  3.53785044,  4.8104698 ,  3.88978364,
        52.68654426,  3.46832544,  2.23701081,  3.11466406,  4.05392477,
         3.90003951,  4.52906532, 

In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 58.33%
The empirical coverage after calibration is: 100.00%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

        0.05         0.1        0.15         0.2         0.3         0.4  \
0  64.848500   72.851300   64.140437   64.042613   77.498963   84.078930   
1  86.882176  110.768428  113.672552  117.271718  139.628736  131.489558   
2  85.322003   97.552222   88.981331   86.942458   86.801024   86.639796   
3  84.894454  120.529428  114.406915  138.211045  175.933187  224.667192   
4  86.288910  119.283996  117.440780  139.348079  163.320748  160.928387   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   81.764112   90.556915   95.273964  110.080139  117.309610  128.831073   
1  138.965581  141.990490  143.996926  136.566099  140.724643  141.846660   
2   91.905869   98.992577   98.609542   98.576721  108.681193  120.971375   
3  237.803198  252.786041  264.178802  268.958503  270.997258  268.721407   
4  164.857213  172.454429  163.011541  164.514715  164.594317  158.621513   

         0.95  Actual  
0  184.151425   73.07  
1  157.106320  141.67  
2  151.7

In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 82.98%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 47 of 47 rows (100.0%)


{'empirical_quantiles': [np.float64(0.02127659574468085),
  np.float64(0.02127659574468085),
  np.float64(0.0425531914893617),
  np.float64(0.06382978723404255),
  np.float64(0.10638297872340426),
  np.float64(0.1702127659574468),
  np.float64(0.1702127659574468),
  np.float64(0.2553191489361702),
  np.float64(0.3191489361702128),
  np.float64(0.40425531914893614),
  np.float64(0.5319148936170213),
  np.float64(0.574468085106383),
  np.float64(0.851063829787234)],
 'average_interval_lengths': [np.float64(35.120941391691446),
  np.float64(58.37860355284053),
  np.float64(97.65875878065333)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.496916020107669),
 'average_crps': np.float64(14.553754629631925),
 'nll_values': array([4.56199759, 3.98785742, 4.12778148, 5.17072267, 5.44819326,
        5.89524988, 4.49150546, 4.27702109, 4.93671491, 3.96048928,
        5.15097247, 4.17957014, 4.04040362, 4.36657484, 4.44139115,
        4.38182738, 4.39040419, 4.56267625, 4.49379

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 75.00%
The empirical coverage after calibration is: 100.00%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())


Early stopping occurred at epoch 47 with best_epoch = 17 and best_val_0_pinball_0.05 = 3.35738



Early stopping occurred at epoch 57 with best_epoch = 27 and best_val_0_pinball_0.1 = 6.04799



Early stopping occurred at epoch 76 with best_epoch = 46 and best_val_0_pinball_0.15 = 5.39592



Early stopping occurred at epoch 48 with best_epoch = 18 and best_val_0_pinball_0.2 = 9.65359



Early stopping occurred at epoch 81 with best_epoch = 51 and best_val_0_pinball_0.3 = 13.28481



Early stopping occurred at epoch 100 with best_epoch = 70 and best_val_0_pinball_0.4 = 10.71168


... (3 outputs left out)


Early stopping occurred at epoch 79 with best_epoch = 49 and best_val_0_pinball_0.8 = 8.71751



Early stopping occurred at epoch 106 with best_epoch = 76 and best_val_0_pinball_0.85 = 9.141



Early stopping occurred at epoch 79 with best_epoch = 49 and best_val_0_pinball_0.9 = 8.12489



Early stopping occurred at epoch 76 with best_epoch = 46 and best_val_0_pinball_0.95 = 4.00316
         0.05         0.1        0.15         0.2         0.3         0.4  \
0   56.718525   98.097694   67.125870   66.435448   94.891632   88.933861   
1   40.743896   76.416260   88.538025   74.090103   98.573380  115.006004   
2   60.845886   66.369354  104.855499   74.637230  127.877968  101.718140   
3  100.382202  131.406845  250.232895  163.871826  321.732300  340.190063   
4  108.668365  136.209015  289.832855   93.271408  195.250031  147.175156   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   97.635780  118.965927   93.647179  214.195068  106.593262  105.144539   
1  113.402000  108.042679  121.650070  107.633240  117.201591  126.287445   
2  148.834045  160.723816  122.054886  227.898727  129.806198  140.192078   
3  305.500885  321.410370  320.006226  358.413544  273.661896  387.095520   
4  281.230591  194.622437  386.410278  154.201675  324.9

In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 89.36%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 47 of 47 rows (100.0%)


{'empirical_quantiles': [np.float64(0.02127659574468085),
  np.float64(0.02127659574468085),
  np.float64(0.0425531914893617),
  np.float64(0.0851063829787234),
  np.float64(0.1276595744680851),
  np.float64(0.2127659574468085),
  np.float64(0.3404255319148936),
  np.float64(0.44680851063829785),
  np.float64(0.574468085106383),
  np.float64(0.6382978723404256),
  np.float64(0.7021276595744681),
  np.float64(0.7872340425531915),
  np.float64(0.9148936170212766)],
 'average_interval_lengths': [np.float64(50.971235640505526),
  np.float64(107.90210496618393),
  np.float64(152.49838711353058)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.87743455001576),
 'average_crps': np.float64(17.067836267917716),
 'nll_values': array([4.91909467, 5.01333544, 4.90541163, 5.44781959, 5.4121605 ,
        6.02171961, 4.28242778, 4.2648311 , 5.53318458, 4.82631917,
        5.06557043, 4.08831548, 5.05456306, 4.46612442, 5.35977359,
        5.22983096, 4.74259793, 4.56436145, 4.97353

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 83.33%
The empirical coverage after calibration is: 91.67%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU
Estimator 0/300, Train metric: 55.0218


Estimator 1/300, Train metric: 53.3739
Estimator 2/300, Train metric: 52.0749
Estimator 3/300, Train metric: 50.5340
Estimator 4/300, Train metric: 48.7228
Estimator 5/300, Train metric: 47.0657
Estimator 6/300, Train metric: 46.0698
Estimator 7/300, Train metric: 44.7027
Estimator 8/300, Train metric: 43.7969
Estimator 9/300, Train metric: 42.3529
Estimator 10/300, Train metric: 41.1388
Estimator 11/300, Train metric: 39.9451
Estimator 12/300, Train metric: 39.3936
Estimator 13/300, Train metric: 38.5708
Estimator 14/300, Train metric: 37.4268
Estimator 15/300, Train metric: 36.2130
Estimator 16/300, Train metric: 35.8212
Estimator 17/300, Train metric: 35.3588
Estimator 18/300, Train metric: 34.8089
Estimator 19/300, Train metric: 34.4374
Estimator 20/300, Train metric: 34.0438
Estimator 21/300, Train metric: 33.6476
Estimator 22/300, Train metric: 32.8975
Estimator 23/300, Train metric: 32.2363
Estimator 24/300, Train metric: 31.7545
Estimator 25/300, Train metric: 31.0023
Estimator

Estimator 48/300, Train metric: 19.5589
Estimator 49/300, Train metric: 19.4191
Estimator 50/300, Train metric: 19.0989
Estimator 51/300, Train metric: 18.7039
Estimator 52/300, Train metric: 18.7178
Estimator 53/300, Train metric: 18.5688
Estimator 54/300, Train metric: 18.3391
Estimator 55/300, Train metric: 17.9783
Estimator 56/300, Train metric: 17.8861
Estimator 57/300, Train metric: 17.7476
Estimator 58/300, Train metric: 17.4496
Estimator 59/300, Train metric: 17.1360
Estimator 60/300, Train metric: 16.9987
Estimator 61/300, Train metric: 16.7382
Estimator 62/300, Train metric: 16.5740
Estimator 63/300, Train metric: 16.3673
Estimator 64/300, Train metric: 16.3097
Estimator 65/300, Train metric: 16.0251
Estimator 66/300, Train metric: 15.7774
Estimator 67/300, Train metric: 15.7003
Estimator 68/300, Train metric: 15.4424
Estimator 69/300, Train metric: 15.2547
Estimator 70/300, Train metric: 15.1427
Estimator 71/300, Train metric: 15.0232
Estimator 72/300, Train metric: 14.7207


Estimator 92/300, Train metric: 12.5994
Estimator 93/300, Train metric: 12.4648
Estimator 94/300, Train metric: 12.2250
Estimator 95/300, Train metric: 12.0125
Estimator 96/300, Train metric: 11.9530
Estimator 97/300, Train metric: 11.9207
Estimator 98/300, Train metric: 11.8422
Estimator 99/300, Train metric: 11.7715
Estimator 100/300, Train metric: 11.7421
Estimator 101/300, Train metric: 11.6483
Estimator 102/300, Train metric: 11.4578
Estimator 103/300, Train metric: 11.3104
Estimator 104/300, Train metric: 11.1709
Estimator 105/300, Train metric: 11.1320
Estimator 106/300, Train metric: 11.1323
Estimator 107/300, Train metric: 11.1133
Estimator 108/300, Train metric: 11.0399
Estimator 109/300, Train metric: 10.8530
Estimator 110/300, Train metric: 10.7520
Estimator 111/300, Train metric: 10.6698
Estimator 112/300, Train metric: 10.6707
Estimator 113/300, Train metric: 10.5762
Estimator 114/300, Train metric: 10.5468
Estimator 115/300, Train metric: 10.4995
Estimator 116/300, Train

Estimator 135/300, Train metric: 9.4091
Estimator 136/300, Train metric: 9.4051
Estimator 137/300, Train metric: 9.2752
Estimator 138/300, Train metric: 9.1582
Estimator 139/300, Train metric: 9.1144
Estimator 140/300, Train metric: 9.1211
Estimator 141/300, Train metric: 9.0928
Estimator 142/300, Train metric: 9.0618
Estimator 143/300, Train metric: 9.0502
Estimator 144/300, Train metric: 8.9818
Estimator 145/300, Train metric: 8.9173
Estimator 146/300, Train metric: 8.8772
Estimator 147/300, Train metric: 8.7642
Estimator 148/300, Train metric: 8.7541
Estimator 149/300, Train metric: 8.7641
Estimator 150/300, Train metric: 8.7435
Estimator 151/300, Train metric: 8.6394
Estimator 152/300, Train metric: 8.6179
Estimator 153/300, Train metric: 8.5644
Estimator 154/300, Train metric: 8.5225
Estimator 155/300, Train metric: 8.3490
Estimator 156/300, Train metric: 8.3194
Estimator 157/300, Train metric: 8.2799
Estimator 158/300, Train metric: 8.2496
Estimator 159/300, Train metric: 8.2563


Estimator 175/300, Train metric: 7.8278
Estimator 176/300, Train metric: 7.8433
Estimator 177/300, Train metric: 7.8214
Estimator 178/300, Train metric: 7.8264
Estimator 179/300, Train metric: 7.7773
Estimator 180/300, Train metric: 7.6972
Estimator 181/300, Train metric: 7.6182
Estimator 182/300, Train metric: 7.5688
Estimator 183/300, Train metric: 7.5420
Estimator 184/300, Train metric: 7.4684
Estimator 185/300, Train metric: 7.4722
Estimator 186/300, Train metric: 7.4772
Estimator 187/300, Train metric: 7.3982
Estimator 188/300, Train metric: 7.3940
Estimator 189/300, Train metric: 7.3914
Estimator 190/300, Train metric: 7.3487
Estimator 191/300, Train metric: 7.2955
Estimator 192/300, Train metric: 7.2783
Estimator 193/300, Train metric: 7.2857
Estimator 194/300, Train metric: 7.2736
Estimator 195/300, Train metric: 7.2801
Estimator 196/300, Train metric: 7.2204
Estimator 197/300, Train metric: 7.1948
Estimator 198/300, Train metric: 7.1825
Estimator 199/300, Train metric: 7.1782


Estimator 217/300, Train metric: 6.7405
Estimator 218/300, Train metric: 6.6536
Estimator 219/300, Train metric: 6.6429
Estimator 220/300, Train metric: 6.6378
Estimator 221/300, Train metric: 6.6014
Estimator 222/300, Train metric: 6.6011
Estimator 223/300, Train metric: 6.6056
Estimator 224/300, Train metric: 6.5337
Estimator 225/300, Train metric: 6.4972
Estimator 226/300, Train metric: 6.4960
Estimator 227/300, Train metric: 6.4901
Estimator 228/300, Train metric: 6.4128
Estimator 229/300, Train metric: 6.3822
Estimator 230/300, Train metric: 6.3718
Estimator 231/300, Train metric: 6.3704
Estimator 232/300, Train metric: 6.3162
Estimator 233/300, Train metric: 6.3248
Estimator 234/300, Train metric: 6.3246
Estimator 235/300, Train metric: 6.3318
Estimator 236/300, Train metric: 6.3256
Estimator 237/300, Train metric: 6.3282
Estimator 238/300, Train metric: 6.3271
Estimator 239/300, Train metric: 6.3365
Estimator 240/300, Train metric: 6.2886
Estimator 241/300, Train metric: 6.2900


Estimator 258/300, Train metric: 5.9267
Estimator 259/300, Train metric: 5.8744
Estimator 260/300, Train metric: 5.8320
Estimator 261/300, Train metric: 5.8273
Estimator 262/300, Train metric: 5.8217
Estimator 263/300, Train metric: 5.8125
Estimator 264/300, Train metric: 5.8159
Estimator 265/300, Train metric: 5.8148
Estimator 266/300, Train metric: 5.7728
Estimator 267/300, Train metric: 5.7383
Estimator 268/300, Train metric: 5.7390
Estimator 269/300, Train metric: 5.7183
Estimator 270/300, Train metric: 5.6924
Estimator 271/300, Train metric: 5.6954
Estimator 272/300, Train metric: 5.6684
Estimator 273/300, Train metric: 5.6512
Estimator 274/300, Train metric: 5.6027
Estimator 275/300, Train metric: 5.5584
Estimator 276/300, Train metric: 5.5620
Estimator 277/300, Train metric: 5.5652
Estimator 278/300, Train metric: 5.5403
Estimator 279/300, Train metric: 5.5374
Estimator 280/300, Train metric: 5.5418
Estimator 281/300, Train metric: 5.5485
Estimator 282/300, Train metric: 5.5233


Estimator 299/300, Train metric: 5.1422
         0.05         0.1        0.15         0.2         0.3         0.4  \
0   89.368225   89.712395   89.884720   90.171989   90.389870   90.629974   
1  144.479233  144.579300  144.624924  144.643768  144.674820  144.697357   
2   98.188240   98.245857   98.256577   98.268898   98.295525   98.308388   
3  265.277527  265.480011  266.200867  266.369965  266.852905  267.042664   
4  164.376160  164.866272  165.195801  165.458145  165.614960  165.836090   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   90.748276   90.925255   91.048500   91.379402   91.630531   91.847900   
1  144.707794  144.732162  144.761765  144.819336  144.833145  144.863419   
2   98.317719   98.333893   98.358498   98.378815   98.383484   98.409103   
3  267.218536  267.382019  267.530884  267.831787  267.954651  268.276886   
4  165.930878  166.118423  166.223175  166.377274  166.622299  166.884384   

         0.95  Actual  
0   92.517

In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [ 73.07 141.67 110.44 275.01 202.57]
Coverage of 90% prediction interval: 4.26%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 47 rows (0.0%)


{'empirical_quantiles': [np.float64(0.3191489361702128),
  np.float64(0.3404255319148936),
  np.float64(0.3404255319148936),
  np.float64(0.3404255319148936),
  np.float64(0.3404255319148936),
  np.float64(0.3404255319148936),
  np.float64(0.3404255319148936),
  np.float64(0.3404255319148936),
  np.float64(0.3617021276595745),
  np.float64(0.3617021276595745),
  np.float64(0.3617021276595745),
  np.float64(0.3617021276595745),
  np.float64(0.3617021276595745)],
 'average_interval_lengths': [np.float64(0.7675105967420213),
  np.float64(1.3415613377347906),
  np.float64(1.895979698668135)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2226.094768181846),
 'average_crps': np.float64(15.39908408785881),
 'nll_values': array([1.71396495e+02, 2.61848661e+02, 9.03078923e+03, 3.15317128e+01,
        9.09498196e+02, 9.30539838e+02, 6.76606748e+02, 1.64754689e+03,
        4.90704587e+01, 1.28292415e+03, 6.34504411e+02, 8.88081181e+03,
        4.17942127e+02, 1.83785129e+01, 1.

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 4.17%
The empirical coverage after calibration is: 95.83%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/quantile_regression/PGBM.xlsx")